# 01 — Exploratory Data Analysis
**EduForecast** | OULAD Dataset | Notebook 1 of 6

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # non-interactive backend — no display required
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Make src/ importable from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DIR, FIGURES_DIR, METRICS_DIR

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

DPI = 150  # fast save; switch to 300 for final report

print('Environment ready.')
print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data     : {RAW_DIR}')
print(f'Figures      : {FIGURES_DIR}')

Environment ready.
Project root : /Users/sergazieff.n/VSCodeProjects/eduforecast
Raw data     : /Users/sergazieff.n/VSCodeProjects/eduforecast/data/raw
Figures      : /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures


## 1. Load Data

In [2]:
print('Loading data...')

tables = {
    'studentInfo':         pd.read_csv(RAW_DIR / 'studentInfo.csv'),
    'studentRegistration': pd.read_csv(RAW_DIR / 'studentRegistration.csv'),
    'studentVle':          pd.read_csv(RAW_DIR / 'studentVle.csv'),
    'assessments':         pd.read_csv(RAW_DIR / 'assessments.csv'),
    'studentAssessment':   pd.read_csv(RAW_DIR / 'studentAssessment.csv'),
    'courses':             pd.read_csv(RAW_DIR / 'courses.csv'),
    'vle':                 pd.read_csv(RAW_DIR / 'vle.csv'),
}

print()
print(f'{"Table":<25} {"Shape":<20} Columns')
print('-' * 80)
for name, df in tables.items():
    print(f'{name:<25} {str(df.shape):<20} {list(df.columns)}')

# Convenient aliases
si  = tables['studentInfo']
sr  = tables['studentRegistration']
sv  = tables['studentVle']
ass = tables['assessments']
sa  = tables['studentAssessment']
co  = tables['courses']
vle = tables['vle']

print('\nData loaded successfully.')

Loading data...



Table                     Shape                Columns
--------------------------------------------------------------------------------
studentInfo               (32593, 12)          ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']
studentRegistration       (32593, 5)           ['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']
studentVle                (10655280, 6)        ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']
assessments               (206, 6)             ['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']
studentAssessment         (173912, 5)          ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']
courses                   (22, 3)              ['code_module', 'code_presentation', 'module_presen

## 2. Dataset Overview

In [3]:
print('=== studentInfo — first 5 rows ===')
print(si.head(5).to_string())

print('\n=== Column data types ===')
print(si.dtypes.to_string())

print('\n=== Missing values ===')
print(si.isnull().sum().to_string())

=== studentInfo — first 5 rows ===
  code_module code_presentation  id_student gender                region      highest_education imd_band age_band  num_of_prev_attempts  studied_credits disability final_result
0         AAA             2013J       11391      M   East Anglian Region       HE Qualification  90-100%     55<=                     0              240          N         Pass
1         AAA             2013J       28400      F              Scotland       HE Qualification   20-30%    35-55                     0               60          N         Pass
2         AAA             2013J       30268      F  North Western Region  A Level or Equivalent   30-40%    35-55                     0               60          Y    Withdrawn
3         AAA             2013J       31604      F     South East Region  A Level or Equivalent   50-60%    35-55                     0               60          N         Pass
4         AAA             2013J       32885      F  West Midlands Region     Low

In [4]:
print('=== code_module value counts ===')
print(si['code_module'].value_counts().to_string())

print('\n=== code_presentation value counts ===')
print(si['code_presentation'].value_counts().to_string())

print('\n=== final_result value counts ===')
print(si['final_result'].value_counts().to_string())

=== code_module value counts ===
code_module
BBB    7909
FFF    7762
DDD    6272
CCC    4434
EEE    2934
GGG    2534
AAA     748

=== code_presentation value counts ===
code_presentation
2014J    11260
2013J     8845
2014B     7804
2013B     4684

=== final_result value counts ===
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024


In [5]:
n_students    = si['id_student'].nunique()
n_modules     = si['code_module'].nunique()
n_vle         = len(sv)
n_assessments = len(ass)

print('=== Basic Statistics ===')
print(f'Total unique students    : {n_students:>8,}')
print(f'Total unique modules     : {n_modules:>8,}')
print(f'Total VLE interactions   : {n_vle:>8,}')
print(f'Total assessment records : {n_assessments:>8,}')
print(f'Total registrations      : {len(sr):>8,}')

=== Basic Statistics ===
Total unique students    :   28,785
Total unique modules     :        7
Total VLE interactions   : 10,655,280
Total assessment records :      206
Total registrations      :   32,593


## 3. Target Variable Distribution

In [6]:
print('Creating target variable...')

si = si.copy()
si['is_at_risk'] = (si['final_result'] == 'Withdrawn').astype(int)

n_at_risk     = si['is_at_risk'].sum()
n_not_at_risk = len(si) - n_at_risk
pct_at_risk   = n_at_risk / len(si) * 100

print(f'At-risk (Withdrawn) : {n_at_risk:,}  ({pct_at_risk:.1f}%)')
print(f'Not at-risk         : {n_not_at_risk:,}  ({100 - pct_at_risk:.1f}%)')

Creating target variable...
At-risk (Withdrawn) : 10,156  (31.2%)
Not at-risk         : 22,437  (68.8%)


In [7]:
print('Creating plots...')

# ── Plot 1: Bar chart — outcome distribution ───────────────────────────────
outcome_order  = ['Pass', 'Distinction', 'Fail', 'Withdrawn']
outcome_colors = {'Pass': '#2ecc71', 'Distinction': '#27ae60',
                  'Fail': '#e67e22', 'Withdrawn': '#e74c3c'}

counts = si['final_result'].value_counts().reindex(outcome_order).fillna(0).astype(int)
total  = counts.sum()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    counts.index,
    counts.values,
    color=[outcome_colors[o] for o in counts.index],
    edgecolor='white',
    linewidth=1.2,
)
for bar, val in zip(bars, counts.values):
    pct = val / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=10,
    )
ax.set_title('Distribution of Student Outcomes', fontsize=14, pad=12)
ax.set_xlabel('Final Result')
ax.set_ylabel('Number of Students')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, counts.max() * 1.18)
plt.tight_layout()
out1 = FIGURES_DIR / '01_outcome_distribution.png'
fig.savefig(out1, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out1}')

Creating plots...
Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_outcome_distribution.png


In [8]:
# ── Plot 2: Pie chart — at-risk split ─────────────────────────────────────
pie_values = [n_not_at_risk, n_at_risk]
pie_labels = ['Not At-Risk', 'At-Risk (Withdrawn)']
pie_colors = ['#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts, autotexts = ax.pie(
    pie_values,
    labels=pie_labels,
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=140,
    explode=(0, 0.06),
    textprops={'fontsize': 12},
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
ax.set_title('At-Risk vs Not At-Risk Students', fontsize=14, pad=10)
plt.tight_layout()
out2 = FIGURES_DIR / '01_atrisk_pie.png'
fig.savefig(out2, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out2}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_atrisk_pie.png


In [9]:
# ── Plot 3: Grouped bar chart — outcomes by module ─────────────────────────
module_outcome = (
    si.groupby(['code_module', 'final_result'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=outcome_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 6))
module_outcome.plot(
    kind='bar',
    ax=ax,
    color=[outcome_colors[o] for o in module_outcome.columns],
    edgecolor='white',
    linewidth=0.8,
)
ax.set_title('Student Outcomes by Module', fontsize=14, pad=12)
ax.set_xlabel('Module')
ax.set_ylabel('Number of Students')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Final Result', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
out3 = FIGURES_DIR / '01_outcomes_by_module.png'
fig.savefig(out3, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved \u2192 {out3}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_outcomes_by_module.png


## 4. VLE Activity Analysis

In [10]:
print('Preparing VLE activity data...')

# Merge VLE interactions with student outcomes
sv_merged = sv.merge(
    si[['id_student', 'code_module', 'code_presentation', 'final_result', 'is_at_risk']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='left',
)

# Map days to weeks (OULAD 'date' is relative to module start)
sv_merged['week'] = np.ceil(sv_merged['date'] / 7).astype(int)

# Keep only weeks 1–39
sv_week = sv_merged[(sv_merged['week'] >= 1) & (sv_merged['week'] <= 39)].copy()

# Total weekly clicks per (student, module, presentation, week)
weekly_student = (
    sv_week
    .groupby(['id_student', 'code_module', 'code_presentation', 'week', 'final_result'])
    ['sum_click'].sum()
    .reset_index()
)

# Average weekly clicks per outcome group
weekly_avg = (
    weekly_student
    .groupby(['week', 'final_result'])['sum_click']
    .mean()
    .reset_index()
)

print(f'VLE records after merge   : {len(sv_merged):,}')
print(f'Weekly student aggregates : {len(weekly_student):,}')

Preparing VLE activity data...


VLE records after merge   : 10,655,280
Weekly student aggregates : 578,863


In [11]:
# ── Plot 4: Line plot — weekly activity by outcome ─────────────────────────
line_cfg = {
    'Withdrawn':  {'color': '#e74c3c', 'ls': '-',  'lw': 2.2},
    'Fail':       {'color': '#e67e22', 'ls': '--', 'lw': 2.0},
    'Pass':       {'color': '#2ecc71', 'ls': '-',  'lw': 2.2},
    'Distinction':{'color': '#1a7a4a', 'ls': '-',  'lw': 2.0},
}

fig, ax = plt.subplots(figsize=(12, 6))
for outcome, cfg in line_cfg.items():
    subset = weekly_avg[weekly_avg['final_result'] == outcome]
    if subset.empty:
        continue
    ax.plot(
        subset['week'], subset['sum_click'],
        color=cfg['color'], linestyle=cfg['ls'],
        linewidth=cfg['lw'], label=outcome, alpha=0.9,
    )

ax.axvline(x=26, color='gray', linestyle=':', linewidth=1.4)
ax.text(26.5, ax.get_ylim()[1] * 0.95, 'Week 26\n(mid-point)',
        color='gray', fontsize=9, va='top')

ax.set_title('Average Weekly VLE Activity by Student Outcome', fontsize=14, pad=12)
ax.set_xlabel('Week Number')
ax.set_ylabel('Average Clicks per Student')
ax.legend(title='Outcome', fontsize=10)
ax.set_xlim(1, 39)
plt.tight_layout()
out4 = FIGURES_DIR / '01_weekly_activity.png'
fig.savefig(out4, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out4}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_weekly_activity.png


In [12]:
# ── Plot 5: Heatmap — avg clicks by module × week (weeks 1–20) ────────────
heatmap_data = (
    sv_week[sv_week['week'] <= 20]
    .groupby(['code_module', 'week'])['sum_click']
    .mean()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.3,
    fmt='.0f',
    cbar_kws={'label': 'Avg Clicks'},
)
ax.set_title('VLE Activity Heatmap by Module (Weeks 1\u201320)', fontsize=14, pad=12)
ax.set_xlabel('Week')
ax.set_ylabel('Module')
plt.tight_layout()
out5 = FIGURES_DIR / '01_activity_heatmap.png'
fig.savefig(out5, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out5}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_activity_heatmap.png


In [13]:
# Key VLE finding
total_clicks_per_student = (
    weekly_student
    .groupby(['id_student', 'final_result'])['sum_click'].sum()
    .reset_index(name='total_clicks')
)

avg_clicks_withdrawn = total_clicks_per_student[
    total_clicks_per_student['final_result'] == 'Withdrawn'
]['total_clicks'].mean()

avg_clicks_pass = total_clicks_per_student[
    total_clicks_per_student['final_result'] == 'Pass'
]['total_clicks'].mean()

# First week where Pass clicks exceed Withdrawn clicks by more than 10%
w_avg = weekly_avg[weekly_avg['final_result'] == 'Withdrawn'].set_index('week')['sum_click']
p_avg = weekly_avg[weekly_avg['final_result'] == 'Pass'].set_index('week')['sum_click']
common_weeks = w_avg.index.intersection(p_avg.index)
denom = p_avg[common_weeks].replace(0, np.nan)
gap_weeks = common_weeks[(p_avg[common_weeks] - w_avg[common_weeks]) / denom > 0.10]
first_gap_week = int(gap_weeks.min()) if len(gap_weeks) > 0 else -1

print(f'Avg total clicks \u2014 Withdrawn : {avg_clicks_withdrawn:,.0f}')
print(f'Avg total clicks \u2014 Pass      : {avg_clicks_pass:,.0f}')
print(f'Click ratio (Pass/Withdrawn) : {avg_clicks_pass / avg_clicks_withdrawn:.2f}x')
print(f'Gap first exceeds 10%%        : week {first_gap_week}')

Avg total clicks — Withdrawn : 462
Avg total clicks — Pass      : 1,900
Click ratio (Pass/Withdrawn) : 4.11x
Gap first exceeds 10%%        : week 1


## 5. Assessment Analysis

In [14]:
print('Preparing assessment data...')

sa_merged = sa.merge(
    ass[['id_assessment', 'date', 'assessment_type', 'weight']],
    on='id_assessment',
    how='left',
).merge(
    si[['id_student', 'final_result', 'code_module', 'code_presentation']],
    on='id_student',
    how='left',
)

sa_merged['score']             = sa_merged['score'].fillna(0)
sa_merged['is_banked']         = sa_merged['is_banked'].fillna(0).astype(int)
sa_merged['submitted_on_time'] = (
    sa_merged['date_submitted'].notna() &
    (sa_merged['date_submitted'] <= sa_merged['date'])
).astype(int)

print(f'Assessment records after merge: {len(sa_merged):,}')

Preparing assessment data...
Assessment records after merge: 207,319


In [15]:
# ── Plot 6: Box plot — score by outcome ────────────────────────────────────
plot_order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
box_palette = {
    'Distinction': '#1a7a4a',
    'Pass':        '#2ecc71',
    'Fail':        '#e67e22',
    'Withdrawn':   '#e74c3c',
}

sa_plot = sa_merged[sa_merged['final_result'].isin(plot_order)].copy()

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=sa_plot,
    x='final_result', y='score',
    order=plot_order,
    palette=box_palette,
    width=0.5,
    fliersize=2,
    ax=ax,
)
ax.set_title('Assessment Score Distribution by Student Outcome', fontsize=14, pad=12)
ax.set_xlabel('Final Result')
ax.set_ylabel('Score (0\u2013100)')
plt.tight_layout()
out6 = FIGURES_DIR / '01_score_distribution.png'
fig.savefig(out6, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out6}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_score_distribution.png


In [16]:
# ── Plot 7: Submission rate by outcome ────────────────────────────────────
sub_rate = (
    sa_merged[sa_merged['final_result'].isin(plot_order)]
    .groupby('final_result')
    .apply(lambda g: g['date_submitted'].notna().mean() * 100)
    .reindex(plot_order)
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    sub_rate.index, sub_rate.values,
    color=[box_palette[o] for o in sub_rate.index],
    edgecolor='white', linewidth=1.2,
)
for bar, val in zip(bars, sub_rate.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.1f}%%',
        ha='center', va='bottom', fontsize=11,
    )
ax.set_title('Assessment Submission Rate by Student Outcome', fontsize=14, pad=12)
ax.set_xlabel('Final Result')
ax.set_ylabel('Submission Rate (%%)')
ax.set_ylim(0, 115)
plt.tight_layout()
out7 = FIGURES_DIR / '01_submission_rate.png'
fig.savefig(out7, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out7}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_submission_rate.png


In [17]:
score_stats = (
    sa_merged[sa_merged['final_result'].isin(plot_order)]
    .groupby('final_result')['score']
    .mean()
    .reindex(plot_order)
)

print('Mean assessment score by outcome:')
for outcome, score in score_stats.items():
    print(f'  {outcome:<12}: {score:.2f}')

print()
print('Assessment submission rate by outcome:')
for outcome, rate in sub_rate.items():
    print(f'  {outcome:<12}: {rate:.1f}%%')

Mean assessment score by outcome:
  Distinction : 88.39
  Pass        : 76.38
  Fail        : 64.79
  Withdrawn   : 68.59

Assessment submission rate by outcome:
  Distinction : 100.0%%
  Pass        : 100.0%%
  Fail        : 100.0%%
  Withdrawn   : 100.0%%


## 6. Correlation Analysis

In [18]:
print('Building per-student summary DataFrame...')

join_keys = ['id_student', 'code_module', 'code_presentation']

click_summary = (
    weekly_student
    .groupby(join_keys)
    .agg(
        total_clicks=('sum_click', 'sum'),
        active_weeks=('week', 'nunique'),
    )
    .reset_index()
)

score_summary = (
    sa_merged
    .groupby('id_student')
    .agg(
        mean_score=('score', 'mean'),
        submission_count=('date_submitted', lambda x: x.notna().sum()),
    )
    .reset_index()
)

target_df = si[join_keys + ['is_at_risk', 'final_result']].drop_duplicates(subset=join_keys)

summary = (
    target_df
    .merge(click_summary, on=join_keys, how='left')
    .merge(score_summary, on='id_student', how='left')
    .fillna(0)
)

print(f'Summary DataFrame: {summary.shape[0]:,} rows x {summary.shape[1]} columns')
print(summary[['total_clicks', 'active_weeks', 'mean_score', 'submission_count', 'is_at_risk']].describe().round(2).to_string())

Building per-student summary DataFrame...


Summary DataFrame: 32,593 rows x 9 columns
       total_clicks  active_weeks  mean_score  submission_count  is_at_risk
count      32593.00      32593.00    32593.00          32593.00    32593.00
mean        1141.09         17.76       59.65              8.58        0.31
std         1612.25         12.88       31.34              9.52        0.46
min            0.00          0.00        0.00              0.00        0.00
25%          116.00          5.00       50.00              2.00        0.00
50%          552.00         18.00       71.20              7.00        0.00
75%         1485.00         30.00       82.20             12.00        1.00
max        23217.00         39.00      100.00             92.00        1.00


In [19]:
# ── Plot 8: Correlation heatmap ───────────────────────────────────────────
num_cols = ['total_clicks', 'active_weeks', 'mean_score', 'submission_count', 'is_at_risk']
corr_matrix = summary[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={'label': 'Pearson r'},
)
ax.set_title('Feature Correlation Matrix (incl. At-Risk Label)', fontsize=13, pad=12)
plt.tight_layout()
out8 = FIGURES_DIR / '01_correlation.png'
fig.savefig(out8, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out8}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_correlation.png


In [20]:
# ── Plot 9: Scatter — total_clicks vs mean_score, coloured by risk ────────
not_risk = summary[summary['is_at_risk'] == 0]
at_risk  = summary[summary['is_at_risk'] == 1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    not_risk['total_clicks'], not_risk['mean_score'],
    c='#2ecc71', alpha=0.35, s=12, label='Not At-Risk',
)
ax.scatter(
    at_risk['total_clicks'], at_risk['mean_score'],
    c='#e74c3c', alpha=0.45, s=12, label='At-Risk (Withdrawn)',
)
ax.set_title('Total Clicks vs Mean Score by Risk Status', fontsize=14, pad=12)
ax.set_xlabel('Total VLE Clicks')
ax.set_ylabel('Mean Assessment Score')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
out9 = FIGURES_DIR / '01_clicks_vs_score.png'
fig.savefig(out9, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out9}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/01_clicks_vs_score.png


## 7. Key Findings Summary

In [21]:
corr_with_target = (
    corr_matrix['is_at_risk']
    .drop('is_at_risk')
    .abs()
    .sort_values(ascending=False)
)
most_predictive = corr_with_target.index[0]

avg_score_withdrawn = float(score_stats.get('Withdrawn', 0))
avg_score_pass      = float(score_stats.get('Pass', 0))
click_ratio = avg_clicks_pass / avg_clicks_withdrawn if avg_clicks_withdrawn > 0 else float('nan')

print('=' * 50)
print('EDA KEY FINDINGS')
print('=' * 50)
print(f'Total students         : {len(si):,}')
print(f'At-risk (Withdrawn)    : {n_at_risk:,} ({pct_at_risk:.1f}%%)')
print(f'Not at-risk            : {n_not_at_risk:,} ({100 - pct_at_risk:.1f}%%)')
print()
print(f'Withdrawn avg clicks   : {avg_clicks_withdrawn:,.0f}')
print(f'Pass avg clicks        : {avg_clicks_pass:,.0f}')
print(f'Click difference       : {click_ratio:.2f}x')
print()
print(f'Withdrawn avg score    : {avg_score_withdrawn:.2f}')
print(f'Pass avg score         : {avg_score_pass:.2f}')
print()
print(f'Most predictive feature: {most_predictive}')
print('=' * 50)

EDA KEY FINDINGS
Total students         : 32,593
At-risk (Withdrawn)    : 10,156 (31.2%%)
Not at-risk            : 22,437 (68.8%%)

Withdrawn avg clicks   : 462
Pass avg clicks        : 1,900
Click difference       : 4.11x

Withdrawn avg score    : 68.59
Pass avg score         : 76.38

Most predictive feature: active_weeks


In [22]:
findings = pd.DataFrame([{
    'total_students':          len(si),
    'n_at_risk':               int(n_at_risk),
    'pct_at_risk':             round(pct_at_risk, 2),
    'n_not_at_risk':           int(n_not_at_risk),
    'avg_clicks_withdrawn':    round(avg_clicks_withdrawn, 1),
    'avg_clicks_pass':         round(avg_clicks_pass, 1),
    'click_ratio':             round(click_ratio, 2),
    'avg_score_withdrawn':     round(avg_score_withdrawn, 2),
    'avg_score_pass':          round(avg_score_pass, 2),
    'most_predictive_feature': most_predictive,
    'first_gap_week':          first_gap_week,
}])

out_csv = METRICS_DIR / 'eda_findings.csv'
findings.to_csv(out_csv, index=False)
print(f'Findings saved \u2192 {out_csv}')

print('\nFigures generated:')
for f in sorted(FIGURES_DIR.glob('01_*.png')):
    print(f'  {f.name}')

Findings saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/metrics/eda_findings.csv

Figures generated:
  01_activity_heatmap.png
  01_atrisk_pie.png
  01_clicks_vs_score.png
  01_correlation.png
  01_outcome_distribution.png
  01_outcomes_by_module.png
  01_score_distribution.png
  01_submission_rate.png
  01_weekly_activity.png
